In [0]:
from pyspark.sql import functions as F
from datetime import datetime



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.landing;
CREATE VOLUME IF NOT EXISTS workspace.landing.customer_files;

In [0]:
SOURCE_TABLE = "workspace.erp.customer"
LANDING_PATH = (
    "/Volumes/workspace/landing/customer_files/customer_cdc"
)

In [0]:
batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

source_df = spark.table(SOURCE_TABLE)

cdc_df = (
    source_df
    .withColumn("_op", F.lit("INSERT"))
    .withColumn("_event_timestamp", F.current_timestamp())
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn("_source_system", F.lit("ERP"))
)

In [0]:
cdc_df.write \
    .format("parquet") \
    .mode("append") \
    .save(f"{LANDING_PATH}/batch_id={batch_id}")

In [0]:
display(
    spark.read
    .format("parquet")
    .load(LANDING_PATH)
)